In [1]:
import napari
import nd2
import cellpose.models as models
import numpy as np
import scipy.ndimage as ndi
import cv2
import plotly.express as px
import pandas as pd
import glob
import skimage as ski

In [2]:
import numpy as np

def filter_labels_near_border(labels, margin=10, in_place=False):
    """
    Remove labeled regions that are within `margin` pixels/voxels of any image border.

    Works for 2D and nD labeled arrays produced by skimage.measure.label.
    Background is assumed to be 0 and is preserved.

    Parameters
    ----------
    labels : np.ndarray
        Labeled image with integer dtype; 0 is background.
    margin : int, optional (default=10)
        Width of the border region (in pixels/voxels) near each image edge.
    in_place : bool, optional (default=False)
        If True, modify `labels` in place; otherwise operate on a copy.

    Returns
    -------
    np.ndarray
        Labeled array with any near-border objects removed (set to 0).
    """
    if not isinstance(labels, np.ndarray):
        raise TypeError("labels must be a numpy ndarray")

    if labels.size == 0 or margin <= 0:
        return labels if in_place else labels.copy()

    lab = labels if in_place else labels.copy()

    # Collect all label ids that appear within `margin` of any border along any axis
    border_labels = set()
    for ax in range(lab.ndim):
        # Lower border slice for this axis
        low_slice = [slice(None)] * lab.ndim
        low_slice[ax] = slice(0, min(margin, lab.shape[ax]))
        border_labels.update(np.unique(lab[tuple(low_slice)]).tolist())

        # Upper border slice for this axis
        high_slice = [slice(None)] * lab.ndim
        start = max(lab.shape[ax] - margin, 0)
        high_slice[ax] = slice(start, lab.shape[ax])
        border_labels.update(np.unique(lab[tuple(high_slice)]).tolist())

    # Remove background from the removal set
    border_labels.discard(0)

    if border_labels:
        # Build a boolean mask for all voxels belonging to border-touching labels
        mask = np.isin(lab, list(border_labels))
        lab[mask] = 0

    return lab

In [3]:
model = models.CellposeModel(gpu=True)

In [6]:
fnames = glob.glob('*/*.nd2')

In [7]:
fname = fnames[0]
img = nd2.ND2File(fname).asarray()
proj_img = np.max(img, axis=1)

C:\Users\smc\AppData\Local\Temp\ipykernel_15460\294212386.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


In [8]:
def process_fname(fname):
    img = nd2.ND2File(fname).asarray()
    print(img.shape)
    proj_img = np.max(img, axis=1)
    ski.io.imsave(fname.replace('.nd2', '.tif'), proj_img)

    masks = np.array([model.eval(proj_img[i,-1])[0] for i in range(proj_img.shape[0])])
    filtered_labels = np.array([filter_labels_near_border(masks[i], margin=10) for i in range(masks.shape[0])])
    ski.io.imsave(fname.replace('.nd2', '.tiff'), filtered_labels)

In [9]:
for fname in fnames:
    process_fname(fname)

C:\Users\smc\AppData\Local\Temp\ipykernel_15460\3057095976.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


(48, 31, 3, 2304, 2304)


C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep1_cen6g18r\perturbed_rep1.tif is a low contrast image
  return func(*args, **kwargs)
C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep1_cen6g18r\perturbed_rep1.tiff is a low contrast image
  return func(*args, **kwargs)
C:\Users\smc\AppData\Local\Temp\ipykernel_15460\3057095976.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


(65, 31, 3, 2304, 2304)


C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep1_cen6g18r\perturbed_rep1_0001.tif is a low contrast image
  return func(*args, **kwargs)
C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep1_cen6g18r\perturbed_rep1_0001.tiff is a low contrast image
  return func(*args, **kwargs)
C:\Users\smc\AppData\Local\Temp\ipykernel_15460\3057095976.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


...

C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3.tif is a low contrast image
  return func(*args, **kwargs)
C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3.tiff is a low contrast image
  return func(*args, **kwargs)
C:\Users\smc\AppData\Local\Temp\ipykernel_15460\3057095976.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


(70, 31, 3, 2304, 2304)


C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3_0001.tif is a low contrast image
  return func(*args, **kwargs)
C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3_0001.tiff is a low contrast image
  return func(*args, **kwargs)
C:\Users\smc\AppData\Local\Temp\ipykernel_15460\3057095976.py:2: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  img = nd2.ND2File(fname).asarray()


(48, 31, 3, 2304, 2304)


C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3_0002.tif is a low contrast image
  return func(*args, **kwargs)
C:\envs\smc\Cellpose4\.pixi\envs\default\Lib\site-packages\skimage\_shared\utils.py:328: UserWarning: rep3_cen6g18r\unperturbed_rep3_0002.tiff is a low contrast image
  return func(*args, **kwargs)
